<a href="https://colab.research.google.com/github/Prashanti2004/Used_Cars_Data_Prep/blob/main/UsedCarsDataPrep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Task 1 — Load, Inspect, and Clean

In [ ]:
import pandas as pd
import numpy as np

url = "/content/used_cars_messy.csv"
df = pd.read_csv(url)

print("Initial Shape:", df.shape)
print(df.info())
print(df.isnull().sum() * 100 / len(df))

Initial Shape: (15661, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15661 entries, 0 to 15660
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15661 non-null  int64  
 1   car_name           15661 non-null  object 
 2   brand              15661 non-null  object 
 3   model              15661 non-null  object 
 4   vehicle_age        15661 non-null  int64  
 5   km_driven          15661 non-null  object 
 6   seller_type        15661 non-null  object 
 7   fuel_type          15661 non-null  object 
 8   transmission_type  15661 non-null  object 
 9   mileage            14376 non-null  object 
 10  engine             13792 non-null  float64
 11  max_power          14862 non-null  float64
 12  seats              15218 non-null  float64
 13  selling_price      15503 non-null  float64
dtypes: float64(4), int64(2), object(8)
memory usage: 1.7+ MB
None
Unnamed: 0            0.00000

In [ ]:
df = df.dropna(subset=['selling_price'])

cols = ['mileage', 'engine', 'max_power']

for col in cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(r'[^0-9.]', '', regex=True)  # remove units
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col].fillna(df[col].median(), inplace=True)

df = df[(df['selling_price'] != 999999999) & (df['selling_price'] >= 10000)]

df = df.drop_duplicates()

print("Shape After Cleaning:", df.shape)

/tmp/ipykernel_17140/3183853785.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
/tmp/ipykernel_17140/3183853785.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', tr

Shape After Cleaning: (15323, 14)


#Task 2 — Encode Categorical Features

In [ ]:
df['transmission_type'] = df['transmission_type'].map({'Manual': 0, 'Automatic': 1})

df = pd.get_dummies(df, columns=['fuel_type', 'seller_type'], drop_first=True)

print(df.columns.tolist())

['Unnamed: 0', 'car_name', 'brand', 'model', 'vehicle_age', 'km_driven', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price', 'fuel_type_ Petrol', 'fuel_type_CNG', 'fuel_type_DIESEL', 'fuel_type_Diesel', 'fuel_type_Electric', 'fuel_type_LPG', 'fuel_type_PETROL', 'fuel_type_Petrol', 'fuel_type_Petrol ', 'fuel_type_diesel', 'fuel_type_diesel ', 'fuel_type_petrol', 'fuel_type_petrol ', 'seller_type_Dealer', 'seller_type_Delaer', 'seller_type_Individual', 'seller_type_Individuall', 'seller_type_Individul', 'seller_type_Trustmark Dealer', 'seller_type_dealer', 'seller_type_individual']


#Task 3 — Split and Compute Baseline MAE

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

X = df.drop(columns=['selling_price'])

X = X.select_dtypes(include=[np.number])

y = df['selling_price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

baseline_pred = np.full(len(y_test), y_train.mean())

mae = mean_absolute_error(y_test, baseline_pred)

print(f"Baseline MAE: ₹{round(mae)}")

Baseline MAE: ₹442943
